In [ ]:
# # This file is run on Google Colab GPU. 
# Colab setup — run this first
!pip install -q xgboost

# Combination Model: FinBERT + Price Features → LinearSVC / XGBoost

**Strategy:** same-day FinBERT sentiment + open gap + previous-day technical indicators → predict intraday direction (close ≥ open).  
No sliding window. All features are known at market open — zero lookahead leakage.  
Direct comparison to StockPredict baseline (FinBERT → LinearSVC = 64.1%).

In [4]:
import os
import shutil
from google.colab import files

WORK_DIR = "/content"
DATA_FILE = f"{WORK_DIR}/sp500_headlines_with_prices.csv"

# Upload data file (only needed once per session)
TARGET_NAME = "sp500_headlines_with_prices.csv"

if not os.path.exists(DATA_FILE):
    print(f"Please upload: {TARGET_NAME}")
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest = os.path.join(WORK_DIR, TARGET_NAME)
        shutil.move(filename, dest)
        print(f"Saved to {dest}")
else:
    print(f"{TARGET_NAME} already exists, skipping upload.")

Please upload: sp500_headlines_with_prices.csv


Saving sp500_headlines_with_prices.csv to sp500_headlines_with_prices.csv
Saved to /content/sp500_headlines_with_prices.csv


In [5]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from xgboost import XGBClassifier
import os, gc, shutil
from google.colab import files

WORK_DIR   = "/content"
DATA_FILE  = f"{WORK_DIR}/sp500_headlines_with_prices.csv"
CACHE_FILE = f"{WORK_DIR}/finbert_probs_combo.npy"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
np.random.seed(42)

Using device: cuda


In [6]:
# 1. Load data & target
df = pd.read_csv(DATA_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df["target"] = (df["close_price"] >= df["open_price"]).astype(int)
print(f"Total rows: {len(df)}")
print(f"Target distribution: {df['target'].value_counts().to_dict()}")

Total rows: 3507
Target distribution: {1: 1898, 0: 1609}


/tmp/ipykernel_815/3174075544.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])


In [7]:
# 2. FinBERT sentiment (cached)
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert   = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
finbert.eval()

def get_finbert_probs(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt",
                           truncation=True, max_length=512, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = finbert(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)

if os.path.exists(CACHE_FILE):
    print("Loading cached FinBERT probs...")
    finbert_probs = np.load(CACHE_FILE)
else:
    print("Running FinBERT inference...")
    texts = df["combined_headline"].fillna("neutral").tolist()
    finbert_probs = get_finbert_probs(texts)
    np.save(CACHE_FILE, finbert_probs)
    print(f"Saved to {CACHE_FILE}")

# {0: positive, 1: negative, 2: neutral}
df["sent_pos"] = finbert_probs[:, 0]
df["sent_neg"] = finbert_probs[:, 1]
df["sent_neu"] = finbert_probs[:, 2]

# Free GPU/MPS memory before the rest
del finbert, tokenizer
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("FinBERT features added. Model freed from memory.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running FinBERT inference...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Saved to /content/finbert_probs_combo.npy
FinBERT features added. Model freed from memory.


In [8]:
# 3. Feature engineering — no lookahead leakage
# All features are computable at/before market open on day i:
#   sent_*        : today's news (available before open)
#   gap_open      : today's open vs yesterday's close (known at open)
#   prev_*        : yesterday's close-based indicators (fully historical)

df["gap_open"] = (df["open_price"] - df["close_price"].shift(1)) / df["close_price"].shift(1)

df["_ma5"]     = df["close_price"].rolling(5).mean()
df["_ma10"]    = df["close_price"].rolling(10).mean()
df["_vol_ma5"] = df["volume"].rolling(5).mean()

def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs  = avg_gain / (avg_loss + 1e-9)
    return (100 - 100 / (1 + rs)) / 100.0

df["_rsi14"] = compute_rsi(df["close_price"], 14)

# Shift by 1: yesterday's values
df["prev_ma5_ratio"]  = df["close_price"].shift(1) / df["_ma5"].shift(1)
df["prev_rsi14"]      = df["_rsi14"].shift(1)
df["prev_vol_ratio"]  = df["volume"].shift(1) / df["_vol_ma5"].shift(1)

FEATURE_COLS = [
    "sent_pos", "sent_neg", "sent_neu",             # same-day FinBERT (3)
    "gap_open",                                      # open gap signal   (1)
    "prev_ma5_ratio", "prev_rsi14", "prev_vol_ratio" # yesterday tech    (3)
]

df = df.dropna(subset=FEATURE_COLS + ["target"]).reset_index(drop=True)
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Rows after dropna: {len(df)}")

Features (7): ['sent_pos', 'sent_neg', 'sent_neu', 'gap_open', 'prev_ma5_ratio', 'prev_rsi14', 'prev_vol_ratio']
Rows after dropna: 3492


In [9]:
# 4. Train / test split (chronological) + scale
split_index = int(len(df) * 0.8)

X_train = df[FEATURE_COLS].iloc[:split_index].values
y_train = df["target"].iloc[:split_index].values
X_test  = df[FEATURE_COLS].iloc[split_index:].values
y_test  = df["target"].iloc[split_index:].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Test target dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")

Train: (2793, 7) | Test: (699, 7)
Test target dist: {np.int64(0): np.int64(330), np.int64(1): np.int64(369)}


In [10]:
# 5. LinearSVC — same classifier as StockPredict, now with 7 features
svc = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=5000, random_state=42))
svc.fit(X_train, y_train)

svc_probs = svc.predict_proba(X_test)[:, 1]
svc_preds = svc.predict(X_test)

print("=== LinearSVC (FinBERT + Price) ===")
print(classification_report(y_test, svc_preds))
print("Accuracy:", round(accuracy_score(y_test, svc_preds), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, svc_probs), 4))

=== LinearSVC (FinBERT + Price) ===
              precision    recall  f1-score   support

           0       0.64      0.57      0.60       330
           1       0.65      0.71      0.68       369

    accuracy                           0.64       699
   macro avg       0.64      0.64      0.64       699
weighted avg       0.64      0.64      0.64       699

Accuracy: 0.6423
ROC-AUC : 0.68


In [11]:
# 6. XGBoost — non-linear alternative
clf = XGBClassifier(
    n_estimators          = 300,
    max_depth             = 3,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    gamma                 = 0.1,
    tree_method           = "hist",
    device                = "cuda" if torch.cuda.is_available() else "cpu",
    random_state          = 42,
    eval_metric           = "logloss",
    early_stopping_rounds = 30,
    verbosity             = 0,
)
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

xgb_probs = clf.predict_proba(X_test)[:, 1]
xgb_preds = (xgb_probs > 0.5).astype(int)

print("\n=== XGBoost (FinBERT + Price) ===")
print(classification_report(y_test, xgb_preds))
print("Accuracy:", round(accuracy_score(y_test, xgb_preds), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, xgb_probs), 4))

[0]	validation_0-logloss:0.68911
[50]	validation_0-logloss:0.65333
[80]	validation_0-logloss:0.65564

=== XGBoost (FinBERT + Price) ===
              precision    recall  f1-score   support

           0       0.65      0.44      0.52       330
           1       0.61      0.79      0.69       369

    accuracy                           0.62       699
   macro avg       0.63      0.61      0.61       699
weighted avg       0.63      0.62      0.61       699

Accuracy: 0.6237
ROC-AUC : 0.6534


In [12]:
# 7. Final comparison
majority = y_test.sum() / len(y_test)
print("============================================")
print(f"  Random baseline (always up)          : {majority:.1%}")
print(f"  StockPredict   FinBERT→LinearSVC     : 64.1%")
print(f"  Combination    LinearSVC (7 features): {accuracy_score(y_test, svc_preds):.1%}")
print(f"  Combination    XGBoost   (7 features): {accuracy_score(y_test, xgb_preds):.1%}")
print("============================================")

  Random baseline (always up)          : 52.8%
  StockPredict   FinBERT→LinearSVC     : 64.1%
  Combination    LinearSVC (7 features): 64.2%
  Combination    XGBoost   (7 features): 62.4%


In [13]:
# 8. Feature importance (XGBoost)
fi = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("--- XGBoost Feature Importance ---")
print(fi.to_string())

# LinearSVC coefficients (relative importance)
coef = pd.Series(svc.calibrated_classifiers_[0].estimator.coef_[0], index=FEATURE_COLS)
print("\n--- LinearSVC Coefficients (+ = bullish signal) ---")
print(coef.sort_values(ascending=False).to_string())

--- XGBoost Feature Importance ---
gap_open          0.260216
sent_pos          0.154440
prev_rsi14        0.138595
sent_neg          0.136177
sent_neu          0.115484
prev_ma5_ratio    0.101947
prev_vol_ratio    0.093142

--- LinearSVC Coefficients (+ = bullish signal) ---
gap_open          0.238166
sent_pos          0.096400
prev_vol_ratio    0.013018
sent_neu          0.007665
prev_rsi14       -0.032799
prev_ma5_ratio   -0.035397
sent_neg         -0.067998
